In [1]:
"""
unit_octonion_phase.py
======================
Gap 5: Unit-octonion phase dynamics.

Derive and verify the discrete differential equation for the
phase of a unit element in the Fano algebra.

Steps:
1. Build Fano algebra (real coefficients, 7 Fano units + identity)
2. Parametrize unit element as spinor
3. Verify 4-pi periodicity (spin-1/2)
4. Test discrete evolution with candidate Delta-theta values
5. Measure closure of the phase orbit
"""

import numpy as np

# =============================================================
# 1. FANO ALGEBRA (real coefficients)
# =============================================================
# Basis: e_0..e_6 (Fano units), 1 (identity)
# We skip t for now (TRB element), because we're looking at
# the spinor phase which lives in the (1, e_i) plane.

lines = [
    (0,1,2), (0,3,4), (0,5,6), (1,3,5), (1,4,6), (2,3,6), (2,4,5)
]

N = 8  # e_0..e_6 + identity at index 7
M = np.zeros((N, N, N))

# Identity
for i in range(N):
    M[7, i, i] = 1
    M[i, 7, i] = 1

# Fano multiplication
for (a, b, c) in lines:
    M[a, b, c] = 1
    M[b, c, a] = 1
    M[c, a, b] = 1
    M[b, a, c] = -1
    M[c, b, a] = -1
    M[a, c, b] = -1

# e_i^2 = -1
for i in range(7):
    M[i, i, 7] = -1

print("=" * 70)
print("STEP 1: Fano algebra constructed")
print("=" * 70)
print(f"Basis: 7 Fano units + identity = {N} elements")
print()


def mul(a, b):
    """Multiply two elements of the algebra."""
    result = np.zeros(N)
    for i in range(N):
        for j in range(N):
            for k in range(N):
                result[k] += a[i] * b[j] * M[i, j, k]
    return result


def norm_sq(a):
    return float(np.dot(a, a))


# Verify: e_0 · e_0 = -1
e0 = np.zeros(N); e0[0] = 1
e0_sq = mul(e0, e0)
print(f"Verify e_0 · e_0 = -1:")
print(f"  e_0 · e_0 = {e0_sq}")
print(f"  Is it -1 (i.e., -identity)? {np.allclose(e0_sq, -np.eye(1, N, 7)[0])}")
print()


# =============================================================
# 2. UNIT-OCTONION PARAMETRIZATION
# =============================================================
print("=" * 70)
print("STEP 2: Unit-octonion phase parametrization")
print("=" * 70)
print()
print("ψ(θ) = cos(θ/2) · 1 + sin(θ/2) · ê")
print()

def psi(theta, e_idx=0):
    """Unit element in the (1, e_idx) plane."""
    v = np.zeros(N)
    v[7] = np.cos(theta / 2)   # identity component
    v[e_idx] = np.sin(theta / 2)  # Fano unit component
    return v


# Verify unit norm
print("Verification of |ψ|² = 1:")
for theta in [0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    p = psi(theta)
    print(f"  θ = {theta:.4f}: |ψ|² = {norm_sq(p):.6f}")

print()

# =============================================================
# 3. 4π PERIODICITY (SPIN-1/2)
# =============================================================
print("=" * 70)
print("STEP 3: Verify 4π periodicity (spin-1/2)")
print("=" * 70)
print()

theta0 = 0.7
p_0 = psi(theta0)
p_2pi = psi(theta0 + 2*np.pi)
p_4pi = psi(theta0 + 4*np.pi)

print(f"ψ(θ) = {p_0}")
print(f"ψ(θ+2π) = {p_2pi}")
print(f"ψ(θ+4π) = {p_4pi}")
print()

print(f"ψ(θ+2π) = -ψ(θ)? {np.allclose(p_2pi, -p_0)}")
print(f"ψ(θ+4π) = ψ(θ)?  {np.allclose(p_4pi, p_0)}")
print()

if np.allclose(p_2pi, -p_0) and np.allclose(p_4pi, p_0):
    print("✓ Spin-1/2 confirmed: ψ has 4π periodicity")
else:
    print("✗ Periodicity check failed")
print()


# =============================================================
# 4. DIFFERENTIAL EQUATION
# =============================================================
print("=" * 70)
print("STEP 4: Differential equation")
print("=" * 70)
print()
print("Continuous: dψ/dθ = (1/2) · ψ · ê")
print()

# Verify numerically
theta = 0.9
h = 1e-6
p_plus = psi(theta + h)
p_minus = psi(theta - h)
dpsi_dtheta = (p_plus - p_minus) / (2 * h)

# Right side: (1/2) ψ · ê
psi_theta = psi(theta)
rhs = 0.5 * mul(psi_theta, e0)

print(f"Numerical dψ/dθ at θ = {theta}:")
print(f"  {dpsi_dtheta}")
print(f"(1/2) ψ · ê:")
print(f"  {rhs}")
print(f"Match? {np.allclose(dpsi_dtheta, rhs, atol=1e-5)}")
print()


# =============================================================
# 5. DISCRETE EVOLUTION AND Δθ
# =============================================================
print("=" * 70)
print("STEP 5: Discrete evolution and the rate Δθ")
print("=" * 70)
print()

# Discrete update: ψ(t+1) = R(Δθ/2) ψ(t)
# i.e., θ advances by Δθ per tick.

def evolve(n_ticks, delta_theta, e_idx=0):
    """Evolve ψ over n_ticks with fixed Δθ."""
    theta = 0.0
    states = [psi(theta, e_idx)]
    for _ in range(n_ticks):
        theta += delta_theta
        states.append(psi(theta, e_idx))
    return states


# Candidates for Δθ based on Finitism numbers
candidates = {
    "Δθ = 4π/137": 4 * np.pi / 137,
    "Δθ = 4π/144": 4 * np.pi / 144,
    "Δθ = 4π/13":  4 * np.pi / 13,
    "Δθ = 4π/7":   4 * np.pi / 7,
    "Δθ = 4π/24":  4 * np.pi / 24,
    "Δθ = 4π/6":   4 * np.pi / 6,
}

print(f"{'Candidate':20s} {'N ticks to 2π':>15s} {'N ticks to 4π':>15s}")
print("-" * 60)

for name, dt in candidates.items():
    # 2π phase closure: total θ = 2π
    n_2pi = 2 * np.pi / dt
    # 4π phase closure: total θ = 4π
    n_4pi = 4 * np.pi / dt
    print(f"{name:20s} {n_2pi:>15.4f} {n_4pi:>15.4f}")

print()
print("Integer N means the phase closes exactly after N ticks.")
print()


# =============================================================
# 6. KEY TEST: WHICH Δθ PRODUCES INTEGER CLOSURE?
# =============================================================
print("=" * 70)
print("STEP 6: Which Δθ produces an integer number of ticks?")
print("=" * 70)
print()

# For the phase to close exactly after N ticks (4π periodicity):
# N · Δθ = 4π
# So Δθ = 4π/N

for N_candidate in [7, 13, 24, 137, 144]:
    dt = 4 * np.pi / N_candidate
    print(f"N = {N_candidate:4d}: Δθ = 4π/{N_candidate} = {dt:.6f}")

print()

# Verify closure
print("Verify closure for each candidate:")
for N_candidate in [7, 13, 24, 137, 144]:
    dt = 4 * np.pi / N_candidate
    states = evolve(N_candidate, dt)
    final = states[-1]
    initial = states[0]
    closed = np.allclose(final, initial, atol=1e-6)
    print(f"  N = {N_candidate:4d}: ψ(N·Δθ) = ψ(0)? {closed}")

print()


# =============================================================
# 7. WHICH N IS NATIVE TO FINITISM?
# =============================================================
print("=" * 70)
print("STEP 7: Native Finitism candidates for N")
print("=" * 70)
print()

native_candidates = {
    "Fano points": 7,
    "13-fold twist": 13,
    "K_max": 24,
    "Fine-structure α⁻¹": 137,
    "Total channels": 144,
    "Registers (1+2+3)": 6,
    "Alphabet size": 10,
}

print(f"{'Source':25s} {'Value':>8s} {'Native?':>10s}")
print("-" * 50)
for name, val in native_candidates.items():
    # Check if this N is native to Finitism
    if name in ["Fano points", "K_max", "Registers (1+2+3)", "Alphabet size"]:
        native = "yes"
    elif name == "Fine-structure α⁻¹":
        native = "derived"
    elif name == "Total channels":
        native = "derived"
    elif name == "13-fold twist":
        native = "derived"
    else:
        native = "?"
    print(f"{name:25s} {val:>8d} {native:>10s}")

print()
print("=" * 70)
print("ANALYSIS")
print("=" * 70)
print()
print("The unit-octonion phase closes after N ticks for Δθ = 4π/N.")
print()
print("For N to be determined by Finitism, N must be a native or")
print("derived Finitism number.")
print()
print("Candidates:")
print("  N = 7   (Fano points)          — native")
print("  N = 137 (fine-structure α⁻¹)   — derived")
print("  N = 13  (13-fold twist)        — derived (from 137 = 13·10 + 7)")
print("  N = 144 (total channels)       — derived")
print()
print("The electron's Compton clock has N = ? ticks per 4π cycle.")
print()

# The Compton period
# The fine-structure constant alpha is the ratio of the electron's
# classical radius to the Compton wavelength (times some factors).
# The Compton period in Planck ticks is roughly:
# T_Compton / t_Planck = (M_Pl / M_e) * 2π ≈ 2π × 4.2 × 10^22
# This is not a clean Finitism number, but if we're doing discrete
# algebra we don't need the physical tick count — we need the
# structural rate.

print("Note on the physical rate:")
print("  The electron's Compton period is ~10^22 Planck ticks,")
print("  which is not a native Finitism number.")
print("  But the STRUCTURAL rate (Δθ per tick) is what matters,")
print("  and it is 4π/N for native N.")
print()
print("The most native choice is N = 7 (Fano points),")
print("giving Δθ = 4π/7 ≈ 1.796 rad/tick.")
print()
print("Alternative: N = 137 (α⁻¹), giving Δθ = 4π/137 ≈ 0.0917 rad/tick.")
print("This connects the phase rate directly to the fine-structure")
print("constant, which is physically appealing but makes N derived.")
print()

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print()
print("✓ Unit-octonion phase defined: ψ(θ) = cos(θ/2) · 1 + sin(θ/2) · ê")
print("✓ 4π periodicity confirmed (spin-1/2)")
print("✓ Differential equation: dψ/dθ = (1/2) ψ · ê")
print("✓ Discrete version: Δθ = 4π/N for integer N")
print()
print("Open: which N is the correct one?")
print("  - N = 7 connects to Fano points (native)")
print("  - N = 137 connects to α⁻¹ (derived)")
print("  - N = 13 connects to the twist (derived from 137)")
print()
print("The structural result is rigorous. The specific N is a")
print("candidate until we can derive it from the algebra E directly.")

STEP 1: Fano algebra constructed
Basis: 7 Fano units + identity = 8 elements

Verify e_0 · e_0 = -1:
  e_0 · e_0 = [ 0.  0.  0.  0.  0.  0.  0. -1.]
  Is it -1 (i.e., -identity)? True

STEP 2: Unit-octonion phase parametrization

ψ(θ) = cos(θ/2) · 1 + sin(θ/2) · ê

Verification of |ψ|² = 1:
  θ = 0.0000: |ψ|² = 1.000000
  θ = 0.7854: |ψ|² = 1.000000
  θ = 1.5708: |ψ|² = 1.000000
  θ = 3.1416: |ψ|² = 1.000000
  θ = 4.7124: |ψ|² = 1.000000
  θ = 6.2832: |ψ|² = 1.000000

STEP 3: Verify 4π periodicity (spin-1/2)

ψ(θ) = [0.34289781 0.         0.         0.         0.         0.
 0.         0.93937271]
ψ(θ+2π) = [-0.34289781  0.          0.          0.          0.          0.
  0.         -0.93937271]
ψ(θ+4π) = [0.34289781 0.         0.         0.         0.         0.
 0.         0.93937271]

ψ(θ+2π) = -ψ(θ)? True
ψ(θ+4π) = ψ(θ)?  True

✓ Spin-1/2 confirmed: ψ has 4π periodicity

STEP 4: Differential equation

Continuous: dψ/dθ = (1/2) · ψ · ê

Numerical dψ/dθ at θ = 0.9:
  [ 0.45022355  0